# 🎯 N-gram Language Models and Smoothing

---

## 📘 Given Corpus

`<s> the cat sat on the mat </s>`  
`<s> the cat sat on the log </s>`  
`<s> the dog sat on the mat </s>`

---

## 🧪 Test Sentence

**S = "the cat sat on the rug"**

---

## 🔍 Questions

---

### 🔹 1. Unigram Model
👉 Compute the **unigram probability** of the sentence **S**

- Show all steps  
- ⚠️ What happens to the unseen word **"rug"**?

---

### 🔹 2. Bigram Model
👉 Compute the **bigram probability** of **S**

- Which bigram gives **zero probability**?  
- ❓ Why does this happen?

---

### 🔹 3. Trigram Model
👉 Compute:

**P(rug | on, the)**

- Does this exist in corpus?  
- ⚠️ What is the impact on sentence probability?

---

## ✨ Smoothing

---

### 🔹 4. Add-1 (Laplace)
👉 Compute:

**P(rug | the)**

- What is **V (vocabulary size)?**  
- Show full calculation  

---

### 🔹 5. Add-k (k = 0.5)
👉 Compute:

**P(rug | the)**

- Compare with Add-1  
- 💡 Which is less aggressive?

---

## 🔁 Advanced Models

---

### 🔹 6. Backoff
👉 Compute:

**P(rug | on, the)**

Fallback order:
> Trigram → Bigram → Unigram

- Why is backoff needed?

---

### 🔹 7. Interpolation

Given:

- λ₁ = 0.5 (Trigram)  
- λ₂ = 0.3 (Bigram)  
- λ₃ = 0.2 (Unigram)

👉 Compute:

**P(rug | on, the)**

- Show contribution of each term  
- 💡 Why better than backoff?

---

### 🔹 8. Discounting

👉 Compute:

**P(mat | the)**  

Given: **d = 0.5**

- What happens to leftover probability?

---

## 🧠 Concept Question

---

### 🔹 9. Which method handles unseen words best?

- Add-1  
- Add-k  
- Backoff  
- Interpolation  
- Discounting  

👉 Justify clearly

---

## 🚀 Extension

---

### 🔹 10. New Sentence

**"the dog sat on the log"**

👉 Perform:

- Unigram  
- Trigram  
- Interpolation  
- Discounting  

---

## ⚡ Key Insight

> Language models don’t fail because they are weak  
> They fail because **data is sparse**

In [14]:
from collections import Counter
import math

corpus = [
    ["<s>", "the", "cat", "sat", "on", "the", "mat", "</s>"],
    ["<s>", "the", "cat", "sat", "on", "the", "log", "</s>"],
    ["<s>", "the", "dog", "sat", "on", "the", "mat", "</s>"]
]

tokens = [w for sent in corpus for w in sent]
vocab = set(tokens)
V = len(vocab)

print(f"Total vocabulary words = {V}")
print(f"No of tokens = {len(tokens)}")
print(f"Vocabulary: {vocab}")
print(f"Tokens: {tokens}")

Total vocabulary words = 9
No of tokens = 24
Vocabulary: {'the', 'mat', '</s>', '<s>', 'cat', 'on', 'sat', 'dog', 'log'}
Tokens: ['<s>', 'the', 'cat', 'sat', 'on', 'the', 'mat', '</s>', '<s>', 'the', 'cat', 'sat', 'on', 'the', 'log', '</s>', '<s>', 'the', 'dog', 'sat', 'on', 'the', 'mat', '</s>']


In [15]:
unigram_counts = Counter(tokens)
unigram_counts

Counter({'<s>': 3,
         'the': 6,
         'cat': 2,
         'sat': 3,
         'on': 3,
         'mat': 2,
         '</s>': 3,
         'log': 1,
         'dog': 1})

In [16]:
bigram_counts = Counter()
trigram_counts = Counter()

for sent in corpus:
    for i in range(len(sent)-1):
        bigram_counts[(sent[i], sent[i+1])] += 1
    for i in range(len(sent)-2):
        trigram_counts[(sent[i], sent[i+1], sent[i+2])] += 1

bigram_counts, trigram_counts

(Counter({('<s>', 'the'): 3,
          ('the', 'cat'): 2,
          ('cat', 'sat'): 2,
          ('sat', 'on'): 3,
          ('on', 'the'): 3,
          ('the', 'mat'): 2,
          ('mat', '</s>'): 2,
          ('the', 'log'): 1,
          ('log', '</s>'): 1,
          ('the', 'dog'): 1,
          ('dog', 'sat'): 1}),
 Counter({('<s>', 'the', 'cat'): 2,
          ('the', 'cat', 'sat'): 2,
          ('cat', 'sat', 'on'): 2,
          ('sat', 'on', 'the'): 3,
          ('on', 'the', 'mat'): 2,
          ('the', 'mat', '</s>'): 2,
          ('on', 'the', 'log'): 1,
          ('the', 'log', '</s>'): 1,
          ('<s>', 'the', 'dog'): 1,
          ('the', 'dog', 'sat'): 1,
          ('dog', 'sat', 'on'): 1}))

### 🔹 1. Unigram Model
👉 Compute the **unigram probability** of the sentence **S**

- Show all steps  
- ⚠️ What happens to the unseen word **"rug"**?

In [18]:
def unigram_prob(w):
    # The count of word 'w' divided by the total number of tokens in the corpus.
    # If 'w' is not in unigram_counts, it defaults to 0.
    return unigram_counts[w] / len(tokens)

sentence = ["the","cat","sat","on","the","rug"]
sentence_prob = 1.0

print("--- Unigram Probabilities for each word ---")
for w in sentence:
    prob = unigram_prob(w)
    print(f"P({w}) = {unigram_counts[w]}/{len(tokens)} = {prob:.4f}")
    sentence_prob *= prob

print("\n--- Handling of unseen word 'rug' ---")
print(f"The word 'rug' does not appear in the corpus. Its count is {unigram_counts['rug']}.")
print(f"Therefore, P(rug) = {unigram_counts['rug']}/{len(tokens)} = {unigram_prob('rug'):.4f}")

print("\n--- Overall Unigram Sentence Probability ---")
print(f"P(S) = P(the) * P(cat) * P(sat) * P(on) * P(the) * P(rug)")
print(f"P(S) = {sentence_prob:.8f}")

if sentence_prob == 0:
    print("\n⚠️ The unigram probability of the sentence is 0 because the word 'rug' is an unseen word in the corpus, resulting in P(rug) = 0. According to the product rule for sentence probability, if any word has a probability of 0, the entire sentence probability becomes 0.")

--- Unigram Probabilities for each word ---
P(the) = 6/24 = 0.2500
P(cat) = 2/24 = 0.0833
P(sat) = 3/24 = 0.1250
P(on) = 3/24 = 0.1250
P(the) = 6/24 = 0.2500
P(rug) = 0/24 = 0.0000

--- Handling of unseen word 'rug' ---
The word 'rug' does not appear in the corpus. Its count is 0.
Therefore, P(rug) = 0/24 = 0.0000

--- Overall Unigram Sentence Probability ---
P(S) = P(the) * P(cat) * P(sat) * P(on) * P(the) * P(rug)
P(S) = 0.00000000

⚠️ The unigram probability of the sentence is 0 because the word 'rug' is an unseen word in the corpus, resulting in P(rug) = 0. According to the product rule for sentence probability, if any word has a probability of 0, the entire sentence probability becomes 0.


### 🔹 2. Bigram Model
👉 Compute the **bigram probability** of **S**

- Which bigram gives **zero probability**?  
- ❓ Why does this happen?

In [19]:
def bigram_prob(w1, w2):
    # P(w2 | w1) = Count(w1, w2) / Count(w1)
    # If Count(w1) is 0, we return 0 to avoid division by zero.
    return bigram_counts[(w1, w2)] / unigram_counts[w1] if unigram_counts[w1] else 0

sentence_with_start = ["<s>"] + sentence # Prepend start token for bigram calculation
sentence_bigram_prob = 1.0
zero_prob_bigrams = []

print("--- Bigram Probabilities for each pair ---")
for i in range(len(sentence_with_start) - 1):
    w1 = sentence_with_start[i]
    w2 = sentence_with_start[i+1]
    prob = bigram_prob(w1, w2)
    print(f"P({w2} | {w1}) = {bigram_counts[(w1, w2)]}/{unigram_counts[w1] if unigram_counts[w1] else 'N/A'} = {prob:.4f}")
    sentence_bigram_prob *= prob
    if prob == 0:
        zero_prob_bigrams.append(f"({w1}, {w2})")

print("\n--- Overall Bigram Sentence Probability ---")
# For simplicity, we typically ignore P(<s>) and calculate P(w_n | w_n-1) * ... * P(w_1 | <s>)
print(f"P(S) = P(the|<s>) * P(cat|the) * P(sat|cat) * P(on|sat) * P(the|on) * P(rug|the)")
print(f"P(S) = {sentence_bigram_prob:.8f}")

print("\n--- Bigram(s) giving zero probability ---")
if zero_prob_bigrams:
    print(f"The following bigram(s) resulted in zero probability: {', '.join(zero_prob_bigrams)}.")
    print("This happens because these specific bigram sequences do not appear in the training corpus.")
    print("⚠️ Consequently, the overall bigram probability of the sentence is 0 because any zero probability component makes the entire product zero.")
else:
    print("No bigrams resulted in zero probability.")

--- Bigram Probabilities for each pair ---
P(the | <s>) = 3/3 = 1.0000
P(cat | the) = 2/6 = 0.3333
P(sat | cat) = 2/2 = 1.0000
P(on | sat) = 3/3 = 1.0000
P(the | on) = 3/3 = 1.0000
P(rug | the) = 0/6 = 0.0000

--- Overall Bigram Sentence Probability ---
P(S) = P(the|<s>) * P(cat|the) * P(sat|cat) * P(on|sat) * P(the|on) * P(rug|the)
P(S) = 0.00000000

--- Bigram(s) giving zero probability ---
The following bigram(s) resulted in zero probability: (the, rug).
This happens because these specific bigram sequences do not appear in the training corpus.
⚠️ Consequently, the overall bigram probability of the sentence is 0 because any zero probability component makes the entire product zero.


### 🔹 3. Trigram Model
👉 Compute:

**P(rug | on, the)**

- Does this exist in corpus?  
- ⚠️ What is the impact on sentence probability?

In [20]:
print("--- Trigram Probability for P(rug | on, the) ---")
w1, w2, w3 = "on", "the", "rug"
prob_trigram = trigram_prob(w1, w2, w3)

print(f"P({w3} | {w1}, {w2}) = {trigram_counts[(w1, w2, w3)]}/{bigram_counts[(w1, w2)] if bigram_counts[(w1, w2)] else 'N/A'} = {prob_trigram:.4f}")

print("\n--- Does this exist in corpus? ---")
if trigram_counts[(w1, w2, w3)] > 0:
    print(f"Yes, the trigram ('{w1}', '{w2}', '{w3}') exists in the corpus {trigram_counts[(w1, w2, w3)]} time(s).")
else:
    print(f"No, the trigram ('{w1}', '{w2}', '{w3}') does NOT exist in the corpus.")

print("\n--- Impact on overall sentence probability ---")
if prob_trigram == 0:
    print("⚠️ If this trigram were part of the test sentence, its zero probability would cause the overall trigram probability of the entire sentence to become 0.")
else:
    print("If this trigram were part of the test sentence, its non-zero probability would contribute to the overall sentence probability.")

--- Trigram Probability for P(rug | on, the) ---
P(rug | on, the) = 0/3 = 0.0000

--- Does this exist in corpus? ---
No, the trigram ('on', 'the', 'rug') does NOT exist in the corpus.

--- Impact on overall sentence probability ---
⚠️ If this trigram were part of the test sentence, its zero probability would cause the overall trigram probability of the entire sentence to become 0.


### 🔹 4. Add-1 (Laplace)
👉 Compute:

**P(rug | the)**

- What is **V (vocabulary size)?**  
- Show full calculation  

In [21]:
def add1_prob(w1, w2):
    # P_add1(w2 | w1) = (Count(w1, w2) + 1) / (Count(w1) + V)
    return (bigram_counts[(w1, w2)] + 1) / (unigram_counts[w1] + V)

w1, w2 = "the", "rug"
count_w1 = unigram_counts[w1] # Count('the')
count_w1w2 = bigram_counts[(w1, w2)] # Count('the', 'rug')

print("--- Add-1 (Laplace) Smoothing for P(rug | the) ---")
print(f"Vocabulary size (V) = {V}")
print(f"Count('{w1}') = {count_w1}")
print(f"Count('{w1}', '{w2}') = {count_w1w2}")

prob_add1 = add1_prob(w1, w2)

print(f"P_add1('{w2}' | '{w1}') = (Count('{w1}', '{w2}') + 1) / (Count('{w1}') + V)")
print(f"P_add1('{w2}' | '{w1}') = ({count_w1w2} + 1) / ({count_w1} + {V})")
print(f"P_add1('{w2}' | '{w1}') = {count_w1w2 + 1} / {count_w1 + V} = {prob_add1:.4f}")

--- Add-1 (Laplace) Smoothing for P(rug | the) ---
Vocabulary size (V) = 9
Count('the') = 6
Count('the', 'rug') = 0
P_add1('rug' | 'the') = (Count('the', 'rug') + 1) / (Count('the') + V)
P_add1('rug' | 'the') = (0 + 1) / (6 + 9)
P_add1('rug' | 'the') = 1 / 15 = 0.0667


### 🔹 5. Add-k (k = 0.5)
👉 Compute:

**P(rug | the)**

- Compare with Add-1  
- 💡 Which is less aggressive?

In [22]:
def addk_prob(w1, w2, k=0.5):
    # P_addk(w2 | w1) = (Count(w1, w2) + k) / (Count(w1) + k * V)
    return (bigram_counts[(w1, w2)] + k) / (unigram_counts[w1] + k * V)

w1, w2 = "the", "rug"
k = 0.5
count_w1 = unigram_counts[w1]
count_w1w2 = bigram_counts[(w1, w2)]

print(f"--- Add-k Smoothing (k={k}) for P(rug | the) ---")
print(f"Vocabulary size (V) = {V}")
print(f"Count('{w1}') = {count_w1}")
print(f"Count('{w1}', '{w2}') = {count_w1w2}")

prob_addk = addk_prob(w1, w2, k)

print(f"P_addk('{w2}' | '{w1}') = (Count('{w1}', '{w2}') + k) / (Count('{w1}') + k * V)")
print(f"P_addk('{w2}' | '{w1}') = ({count_w1w2} + {k}) / ({count_w1} + {k} * {V})")
print(f"P_addk('{w2}' | '{w1}') = {count_w1w2 + k} / {count_w1 + k * V} = {prob_addk:.4f}")

print("\n--- Comparison with Add-1 Smoothing ---")
prob_add1 = (count_w1w2 + 1) / (count_w1 + V) # Re-calculate Add-1 for comparison
print(f"Add-1 (Laplace) P(rug | the) = {prob_add1:.4f}")
print(f"Add-k (k={k}) P(rug | the) = {prob_addk:.4f}")

if prob_addk < prob_add1:
    print(f"💡 With k={k}, Add-k smoothing ({prob_addk:.4f}) is less aggressive than Add-1 smoothing ({prob_add1:.4f}), assigning a lower probability to the unseen event.")
elif prob_addk > prob_add1:
    print(f"💡 With k={k}, Add-k smoothing ({prob_addk:.4f}) is more aggressive than Add-1 smoothing ({prob_add1:.4f}), assigning a higher probability to the unseen event.")
else:
    print(f"💡 With k={k}, Add-k smoothing gives the same probability as Add-1 smoothing.")

--- Add-k Smoothing (k=0.5) for P(rug | the) ---
Vocabulary size (V) = 9
Count('the') = 6
Count('the', 'rug') = 0
P_addk('rug' | 'the') = (Count('the', 'rug') + k) / (Count('the') + k * V)
P_addk('rug' | 'the') = (0 + 0.5) / (6 + 0.5 * 9)
P_addk('rug' | 'the') = 0.5 / 10.5 = 0.0476

--- Comparison with Add-1 Smoothing ---
Add-1 (Laplace) P(rug | the) = 0.0667
Add-k (k=0.5) P(rug | the) = 0.0476
💡 With k=0.5, Add-k smoothing (0.0476) is less aggressive than Add-1 smoothing (0.0667), assigning a lower probability to the unseen event.


### 🔹 6. Backoff
👉 Compute:

**P(rug | on, the)**

Fallback order:
> Trigram → Bigram → Unigram

- Why is backoff needed?

In [23]:
def backoff_prob(w1, w2, w3):
    print(f"Attempting to compute P({w3} | {w1}, {w2}) using Backoff:")
    # Try Trigram first
    if (w1, w2, w3) in trigram_counts and trigram_counts[(w1, w2, w3)] > 0:
        prob = trigram_prob(w1, w2, w3)
        print(f"  Trigram ('{w1}', '{w2}', '{w3}') found. Using P({w3} | {w1}, {w2}) = {prob:.4f}")
        return prob
    else:
        print(f"  Trigram ('{w1}', '{w2}', '{w3}') not found or count is 0. Falling back to Bigram...")
        # Fallback to Bigram
        if (w2, w3) in bigram_counts and bigram_counts[(w2, w3)] > 0:
            prob = bigram_prob(w2, w3)
            print(f"  Bigram ('{w2}', '{w3}') found. Using P({w3} | {w2}) = {prob:.4f}")
            return prob
        else:
            print(f"  Bigram ('{w2}', '{w3}') not found or count is 0. Falling back to Unigram...")
            # Fallback to Unigram
            prob = unigram_prob(w3)
            print(f"  Unigram ('{w3}') used. P({w3}) = {prob:.4f}")
            return prob

w1, w2, w3 = "on", "the", "rug"
backoff_result = backoff_prob(w1, w2, w3)

print(f"\nWhy is backoff needed?")
print(f"Backoff is needed to handle data sparsity, especially when dealing with higher-order N-grams that might not appear in the training corpus. Instead of assigning a zero probability (which would make the entire sentence probability zero), backoff models gracefully degrade to lower-order N-grams (e.g., from trigram to bigram to unigram) to provide a non-zero, albeit less specific, probability estimate. This prevents the 'zero-probability problem' for unseen sequences.")

Attempting to compute P(rug | on, the) using Backoff:
  Trigram ('on', 'the', 'rug') not found or count is 0. Falling back to Bigram...
  Bigram ('the', 'rug') not found or count is 0. Falling back to Unigram...
  Unigram ('rug') used. P(rug) = 0.0000

Why is backoff needed?
Backoff is needed to handle data sparsity, especially when dealing with higher-order N-grams that might not appear in the training corpus. Instead of assigning a zero probability (which would make the entire sentence probability zero), backoff models gracefully degrade to lower-order N-grams (e.g., from trigram to bigram to unigram) to provide a non-zero, albeit less specific, probability estimate. This prevents the 'zero-probability problem' for unseen sequences.


### 🔹 7. Interpolation

Given:

- λ₁ = 0.5 (Trigram)  
- λ₂ = 0.3 (Bigram)  
- λ₃ = 0.2 (Unigram)

👉 Compute:

**P(rug | on, the)**

- Show contribution of each term  
- 💡 Why better than backoff?

In [24]:
def interpolation_prob(w1, w2, w3, l1=0.5, l2=0.3, l3=0.2):
    # Calculate individual probabilities
    p_trigram = trigram_prob(w1, w2, w3)
    p_bigram = bigram_prob(w2, w3) # This is P(w3 | w2)
    p_unigram = unigram_prob(w3)

    # Calculate weighted sum
    interpolated_prob = (l1 * p_trigram) + (l2 * p_bigram) + (l3 * p_unigram)

    print(f"--- Interpolation for P({w3} | {w1}, {w2}) ---")
    print(f"Lambda (λ) values: λ1={l1} (Trigram), λ2={l2} (Bigram), λ3={l3} (Unigram)\n")

    print(f"P_trigram({w3} | {w1}, {w2}) = {p_trigram:.4f}")
    print(f"Contribution from Trigram: {l1} * {p_trigram:.4f} = {l1 * p_trigram:.4f}\n")

    print(f"P_bigram({w3} | {w2}) = {p_bigram:.4f}")
    print(f"Contribution from Bigram: {l2} * {p_bigram:.4f} = {l2 * p_bigram:.4f}\n")

    print(f"P_unigram({w3}) = {p_unigram:.4f}")
    print(f"Contribution from Unigram: {l3} * {p_unigram:.4f} = {l3 * p_unigram:.4f}\n")

    print(f"Interpolated P({w3} | {w1}, {w2}) = {interpolated_prob:.4f}")
    return interpolated_prob

w1, w2, w3 = "on", "the", "rug"
interpolation_result = interpolation_prob(w1, w2, w3)

print("\n--- Why is Interpolation better than Backoff? ---")
print("💡 Interpolation blends probability estimates from different N-gram orders, even when higher-order counts are available. This means it doesn't discard information from lower-order N-grams entirely, which can be beneficial in situations where a higher-order N-gram might have been observed but is not fully reliable (e.g., due to sparse data). In contrast, backoff *falls back* to lower-order models only when higher-order models have zero counts. Interpolation provides a smoother probability distribution, as it always considers some weight from all N-gram orders, potentially leading to more robust estimates, especially for infrequent but observed sequences. However, in cases where all constituent probabilities are zero, like here with 'rug' (since P(rug) is 0, P(rug|the) is 0, and P(rug|on,the) is 0), interpolation will also yield 0 unless a smoothing technique is applied to the individual n-gram probabilities before interpolation. The current implementation uses unsmoothed probabilities for each N-gram term.")

--- Interpolation for P(rug | on, the) ---
Lambda (λ) values: λ1=0.5 (Trigram), λ2=0.3 (Bigram), λ3=0.2 (Unigram)

P_trigram(rug | on, the) = 0.0000
Contribution from Trigram: 0.5 * 0.0000 = 0.0000

P_bigram(rug | the) = 0.0000
Contribution from Bigram: 0.3 * 0.0000 = 0.0000

P_unigram(rug) = 0.0000
Contribution from Unigram: 0.2 * 0.0000 = 0.0000

Interpolated P(rug | on, the) = 0.0000

--- Why is Interpolation better than Backoff? ---
💡 Interpolation blends probability estimates from different N-gram orders, even when higher-order counts are available. This means it doesn't discard information from lower-order N-grams entirely, which can be beneficial in situations where a higher-order N-gram might have been observed but is not fully reliable (e.g., due to sparse data). In contrast, backoff *falls back* to lower-order models only when higher-order models have zero counts. Interpolation provides a smoother probability distribution, as it always considers some weight from all N-gram or

### 🔹 8. Discounting

👉 Compute:

**P(mat | the)**  

Given: **d = 0.5**

- What happens to leftover probability?


In [34]:
def discount_prob(w1, w2, d_value=0.5):
    # P_discounted(w2 | w1) = max(Count(w1, w2) - d, 0) / Count(w1)
    # We use d_value as a parameter name to avoid conflict with global 'd'
    return max(bigram_counts[(w1, w2)] - d_value, 0) / unigram_counts[w1]

w1, w2 = "the", "mat"
d = 0.5

count_w1 = unigram_counts[w1] # Count('the')
count_w1w2 = bigram_counts[(w1, w2)] # Count('the', 'mat')

print(f"--- Bigram Discounting for P('{w2}' | '{w1}') with d={d} ---")
print(f"Count('{w1}') = {count_w1}")
print(f"Count('{w1}', '{w2}') = {count_w1w2}")

prob_discounted = discount_prob(w1, w2, d)

print(f"P_discounted('{w2}' | '{w1}') = (max(Count('{w1}', '{w2}') - d, 0)) / Count('{w1}')")
print(f"P_discounted('{w2}' | '{w1}') = (max({count_w1w2} - {d}, 0)) / {count_w1}")
print(f"P_discounted('{w2}' | '{w1}') = {max(count_w1w2 - d, 0)} / {count_w1} = {prob_discounted:.4f}")

# Calculate the total discounted probability for all observed bigrams starting with 'the'
total_discounted_prob_for_the = 0
print(f"\nDiscounted probabilities for bigrams starting with '{w1}':")
for bigram_tuple, count in bigram_counts.items():
    if bigram_tuple[0] == w1:
        # Original probability (for comparison)
        original_prob = bigram_counts[bigram_tuple] / unigram_counts[w1]
        # Discounted probability
        discounted = discount_prob(w1, bigram_tuple[1], d)
        total_discounted_prob_for_the += discounted
        print(f"  P_discounted({bigram_tuple[1]} | {w1}) = {discounted:.4f} (Original: {original_prob:.4f})")

# Calculate the leftover probability mass
leftover_prob_mass = 1 - total_discounted_prob_for_the

print(f"\n--- What happens to leftover probability mass? ---")
print(f"The sum of discounted probabilities for bigrams starting with '{w1}' is: {total_discounted_prob_for_the:.4f}")
print(f"The remaining probability mass is: 1 - {total_discounted_prob_for_the:.4f} = {leftover_prob_mass:.4f}")
print(f"This 'leftover probability mass' is then typically redistributed among unseen N-grams (or N-grams with counts less than 'd') using other smoothing techniques, such as absolute discounting or Kneser-Ney smoothing. It allows for non-zero probabilities for previously unobserved events while giving a small 'discount' to observed events.")

--- Bigram Discounting for P('mat' | 'the') with d=0.5 ---
Count('the') = 6
Count('the', 'mat') = 2
P_discounted('mat' | 'the') = (max(Count('the', 'mat') - d, 0)) / Count('the')
P_discounted('mat' | 'the') = (max(2 - 0.5, 0)) / 6
P_discounted('mat' | 'the') = 1.5 / 6 = 0.2500

Discounted probabilities for bigrams starting with 'the':
  P_discounted(cat | the) = 0.2500 (Original: 0.3333)
  P_discounted(mat | the) = 0.2500 (Original: 0.3333)
  P_discounted(log | the) = 0.0833 (Original: 0.1667)
  P_discounted(dog | the) = 0.0833 (Original: 0.1667)

--- What happens to leftover probability mass? ---
The sum of discounted probabilities for bigrams starting with 'the' is: 0.6667
The remaining probability mass is: 1 - 0.6667 = 0.3333
This 'leftover probability mass' is then typically redistributed among unseen N-grams (or N-grams with counts less than 'd') using other smoothing techniques, such as absolute discounting or Kneser-Ney smoothing. It allows for non-zero probabilities for previou

### 🔹 9. Which method handles unseen words best?

- Add-1  
- Add-k  
- Backoff  
- Interpolation  
- Discounting  

👉 Justify clearly

 ### 🔹 9. Which method handles unseen words best?

**Best Answer: Interpolation**

---

### Justification

- **Add-1 smoothing**
  - Assigns equal probability to all unseen events  
  - ⚠️ Too aggressive → distorts true distribution  

- **Add-k smoothing**
  - More controlled than Add-1  
  - Still uniformly distributes probability → not context-aware  

- **Backoff**
  - Falls back to lower-order models when data is missing  
  - ⚠️ Discards higher-order information completely  

- **Discounting**
  - Reduces probability of seen events to allocate mass for unseen ones  
  - ⚠️ Needs additional redistribution mechanism to handle unseen words  

---

### Why Interpolation is Best

- Combines **trigram + bigram + unigram simultaneously**  
- Never assigns zero probability  
- Uses **all available information instead of discarding it**  
- More **robust to data sparsity**  

---

### Final Insight

> Interpolation is preferred because it balances higher-order context with lower-order reliability, ensuring stable and realistic probability estimates even for unseen events.

### 🔹 10. New Sentence

**"the dog sat on the log"**

👉 Perform:

- Unigram  
- Trigram  
- Interpolation  
- Discounting  

In [27]:
sentence2 = ["the", "dog", "sat", "on", "the", "log"]
print("Test sentence:", " ".join(sentence2))

Test sentence: the dog sat on the log


In [28]:
## unigram probability of full sentence
print("\n================ UNIGRAM MODEL =================\n")

total_uni = 1

for word in sentence2:
    count_w = unigram_counts[word]
    total_tokens = len(tokens)
    prob = count_w / total_tokens

    print(f"P({word}) = Count({word}) / Total Tokens")
    print(f"       = {count_w} / {total_tokens} = {prob:.4f}\n")

    total_uni *= prob

print(f"Final Unigram Probability = {total_uni:.8f}")


================ UNIGRAM MODEL =================

P(the) = Count(the) / Total Tokens
       = 6 / 24 = 0.2500

P(dog) = Count(dog) / Total Tokens
       = 1 / 24 = 0.0417

P(sat) = Count(sat) / Total Tokens
       = 3 / 24 = 0.1250

P(on) = Count(on) / Total Tokens
       = 3 / 24 = 0.1250

P(the) = Count(the) / Total Tokens
       = 6 / 24 = 0.2500

P(log) = Count(log) / Total Tokens
       = 1 / 24 = 0.0417

Final Unigram Probability = 0.00000170


In [29]:
## bigram probability of full sentence
print("\n================ BIGRAM MODEL =================\n")

total_bi = 1
prev = "<s>"

for word in sentence2:
    count_prev = unigram_counts[prev]
    count_bigram = bigram_counts[(prev, word)]

    prob = count_bigram / count_prev if count_prev > 0 else 0

    print(f"P({word} | {prev}) = Count({prev}, {word}) / Count({prev})")
    print(f"                 = {count_bigram} / {count_prev} = {prob:.4f}\n")

    total_bi *= prob
    prev = word

# include end token
count_prev = unigram_counts[prev]
count_bigram = bigram_counts[(prev, "</s>")]
prob = count_bigram / count_prev if count_prev > 0 else 0

print(f"P(</s> | {prev}) = Count({prev}, </s>) / Count({prev})")
print(f"               = {count_bigram} / {count_prev} = {prob:.4f}\n")

total_bi *= prob

print(f"Final Bigram Probability = {total_bi:.8f}")


================ BIGRAM MODEL =================

P(the | <s>) = Count(<s>, the) / Count(<s>)
                 = 3 / 3 = 1.0000

P(dog | the) = Count(the, dog) / Count(the)
                 = 1 / 6 = 0.1667

P(sat | dog) = Count(dog, sat) / Count(dog)
                 = 1 / 1 = 1.0000

P(on | sat) = Count(sat, on) / Count(sat)
                 = 3 / 3 = 1.0000

P(the | on) = Count(on, the) / Count(on)
                 = 3 / 3 = 1.0000

P(log | the) = Count(the, log) / Count(the)
                 = 1 / 6 = 0.1667

P(</s> | log) = Count(log, </s>) / Count(log)
               = 1 / 1 = 1.0000

Final Bigram Probability = 0.02777778


In [31]:
## Trigram
print("\n================ TRIGRAM MODEL =================\n")

sentence2 = ["the", "dog", "sat", "on", "the", "log"]

total_tri = 1

# Add two start tokens
tokens_with_start = ["<s>", "<s>"] + sentence2 + ["</s>"]

for i in range(2, len(tokens_with_start)):

    w1 = tokens_with_start[i-2]
    w2 = tokens_with_start[i-1]
    w3 = tokens_with_start[i]

    count_tri = trigram_counts[(w1, w2, w3)]
    count_bi = bigram_counts[(w1, w2)]

    prob = count_tri / count_bi if count_bi > 0 else 0

    print(f"🔹 P({w3} | {w1}, {w2})")
    print(f"   Count({w1}, {w2}, {w3}) = {count_tri}")
    print(f"   Count({w1}, {w2}) = {count_bi}")
    print(f"   Probability = {count_tri} / {count_bi} = {prob:.4f}\n")

    total_tri *= prob

print("================ FINAL TRIGRAM PROBABILITY =================")
print(f"P(sentence) = {total_tri:.8f}")


================ TRIGRAM MODEL =================

🔹 P(the | <s>, <s>)
   Count(<s>, <s>, the) = 0
   Count(<s>, <s>) = 0
   Probability = 0 / 0 = 0.0000

🔹 P(dog | <s>, the)
   Count(<s>, the, dog) = 1
   Count(<s>, the) = 3
   Probability = 1 / 3 = 0.3333

🔹 P(sat | the, dog)
   Count(the, dog, sat) = 1
   Count(the, dog) = 1
   Probability = 1 / 1 = 1.0000

🔹 P(on | dog, sat)
   Count(dog, sat, on) = 1
   Count(dog, sat) = 1
   Probability = 1 / 1 = 1.0000

🔹 P(the | sat, on)
   Count(sat, on, the) = 3
   Count(sat, on) = 3
   Probability = 3 / 3 = 1.0000

🔹 P(log | on, the)
   Count(on, the, log) = 1
   Count(on, the) = 3
   Probability = 1 / 3 = 0.3333

🔹 P(</s> | the, log)
   Count(the, log, </s>) = 1
   Count(the, log) = 1
   Probability = 1 / 1 = 1.0000

================ FINAL TRIGRAM PROBABILITY =================
P(sentence) = 0.00000000


In [30]:
## interpolation for full sentence
print("\n================ INTERPOLATION MODEL =================\n")

lambda1 = 0.5   # trigram weight
lambda2 = 0.3   # bigram weight
lambda3 = 0.2   # unigram weight

total_interp = 1
w1, w2 = "<s>", sentence2[0]

for i in range(len(sentence2)):
    w3 = sentence2[i]

    # context for trigram
    if i == 0:
        prev2, prev1 = "<s>", "<s>"
    elif i == 1:
        prev2, prev1 = "<s>", sentence2[0]
    else:
        prev2, prev1 = sentence2[i-2], sentence2[i-1]

    # trigram probability
    tri_count = trigram_counts[(prev2, prev1, w3)]
    bi_context_count = bigram_counts[(prev2, prev1)]
    p_tri = tri_count / bi_context_count if bi_context_count > 0 else 0

    # bigram probability
    bi_count = bigram_counts[(prev1, w3)]
    uni_context_count = unigram_counts[prev1]
    p_bi = bi_count / uni_context_count if uni_context_count > 0 else 0

    # unigram probability
    p_uni = unigram_counts[w3] / len(tokens)

    # interpolation
    p_interp = lambda1 * p_tri + lambda2 * p_bi + lambda3 * p_uni

    print(f"Word: {w3}")
    print(f"  Trigram   = P({w3} | {prev2}, {prev1}) = {p_tri:.4f}")
    print(f"  Bigram    = P({w3} | {prev1}) = {p_bi:.4f}")
    print(f"  Unigram   = P({w3}) = {p_uni:.4f}")
    print(f"  Interpolated = {lambda1}*{p_tri:.4f} + {lambda2}*{p_bi:.4f} + {lambda3}*{p_uni:.4f}")
    print(f"               = {p_interp:.4f}\n")

    total_interp *= p_interp

print(f"Final Interpolation Probability = {total_interp:.8f}")


================ INTERPOLATION MODEL =================

Word: the
  Trigram   = P(the | <s>, <s>) = 0.0000
  Bigram    = P(the | <s>) = 1.0000
  Unigram   = P(the) = 0.2500
  Interpolated = 0.5*0.0000 + 0.3*1.0000 + 0.2*0.2500
               = 0.3500

Word: dog
  Trigram   = P(dog | <s>, the) = 0.3333
  Bigram    = P(dog | the) = 0.1667
  Unigram   = P(dog) = 0.0417
  Interpolated = 0.5*0.3333 + 0.3*0.1667 + 0.2*0.0417
               = 0.2250

Word: sat
  Trigram   = P(sat | the, dog) = 1.0000
  Bigram    = P(sat | dog) = 1.0000
  Unigram   = P(sat) = 0.1250
  Interpolated = 0.5*1.0000 + 0.3*1.0000 + 0.2*0.1250
               = 0.8250

Word: on
  Trigram   = P(on | dog, sat) = 1.0000
  Bigram    = P(on | sat) = 1.0000
  Unigram   = P(on) = 0.1250
  Interpolated = 0.5*1.0000 + 0.3*1.0000 + 0.2*0.1250
               = 0.8250

Word: the
  Trigram   = P(the | sat, on) = 1.0000
  Bigram    = P(the | on) = 1.0000
  Unigram   = P(the) = 0.2500
  Interpolated = 0.5*1.0000 + 0.3*1.0000 + 0.2*0

In [32]:
## Discounting in Bigram
sentence2 = ["the", "dog", "sat", "on", "the", "log"]

print("\n================ CUSTOM 'ADD-1 DISCOUNTING' (FULL SENTENCE) =================\n")

total_prob = 1
prev = "<s>"

for i, word in enumerate(sentence2):

    w1 = prev
    w2 = word

    count_w1 = unigram_counts[w1]
    count_w1w2 = bigram_counts[(w1, w2)]

    print(f"\n🔹 Step {i+1}: P('{w2}' | '{w1}')")
    print(f"Count('{w1}') = {count_w1}")
    print(f"Count('{w1}', '{w2}') = {count_w1w2}")

    # --- Intermediate value (your style) ---
    numerator = (count_w1w2 + 1) * count_w1
    denominator = (count_w1 + V)
    intermediate_value = numerator / denominator

    print("\nIntermediate 'Discounted Value' Calculation:")
    print(f"= (Count('{w1}', '{w2}') + 1) * Count('{w1}') / (Count('{w1}') + V)")
    print(f"= ({count_w1w2} + 1) * {count_w1} / ({count_w1} + {V})")
    print(f"= {count_w1w2 + 1} * {count_w1} / {count_w1 + V}")
    print(f"= {numerator} / {denominator} = {intermediate_value:.4f}")

    # --- Final probability ---
    prob = intermediate_value / count_w1

    print("\nProbability derived from this 'Discounted Value':")
    print(f"P_custom_discount('{w2}' | '{w1}') = {intermediate_value:.4f} / {count_w1}")
    print(f"= {prob:.4f}")

    total_prob *= prob
    prev = word

# Final EOS
w1 = prev
w2 = "</s>"

count_w1 = unigram_counts[w1]
count_w1w2 = bigram_counts[(w1, w2)]

numerator = (count_w1w2 + 1) * count_w1
denominator = (count_w1 + V)
intermediate_value = numerator / denominator

print(f"\n🔹 Final Step: P('</s>' | '{w1}')")

print("\nIntermediate 'Discounted Value' Calculation:")
print(f"= ({count_w1w2} + 1) * {count_w1} / ({count_w1} + {V})")
print(f"= {numerator} / {denominator} = {intermediate_value:.4f}")

prob = intermediate_value / count_w1

print("\nProbability derived from this 'Discounted Value':")
print(f"P_custom_discount('</s>' | '{w1}') = {intermediate_value:.4f} / {count_w1}")
print(f"= {prob:.4f}")

total_prob *= prob

print("\n================ FINAL SENTENCE PROBABILITY =================")
print(f"P(sentence) = {total_prob:.6f}")


================ CUSTOM 'ADD-1 DISCOUNTING' (FULL SENTENCE) =================


🔹 Step 1: P('the' | '<s>')
Count('<s>') = 3
Count('<s>', 'the') = 3

Intermediate 'Discounted Value' Calculation:
= (Count('<s>', 'the') + 1) * Count('<s>') / (Count('<s>') + V)
= (3 + 1) * 3 / (3 + 9)
= 4 * 3 / 12
= 12 / 12 = 1.0000

Probability derived from this 'Discounted Value':
P_custom_discount('the' | '<s>') = 1.0000 / 3
= 0.3333

🔹 Step 2: P('dog' | 'the')
Count('the') = 6
Count('the', 'dog') = 1

Intermediate 'Discounted Value' Calculation:
= (Count('the', 'dog') + 1) * Count('the') / (Count('the') + V)
= (1 + 1) * 6 / (6 + 9)
= 2 * 6 / 15
= 12 / 15 = 0.8000

Probability derived from this 'Discounted Value':
P_custom_discount('dog' | 'the') = 0.8000 / 6
= 0.1333

🔹 Step 3: P('sat' | 'dog')
Count('dog') = 1
Count('dog', 'sat') = 1

Intermediate 'Discounted Value' Calculation:
= (Count('dog', 'sat') + 1) * Count('dog') / (Count('dog') + V)
= (1 + 1) * 1 / (1 + 9)
= 2 * 1 / 10
= 2 / 10 = 0.2000

Pro

In [33]:
print("\n================ FINAL COMPARISON =================\n")

print(f"Unigram Probability            : {total_uni:.8f}")
print(f"Bigram Probability             : {total_bi:.8f}")
print(f"Trigram Probability            : {total_tri:.8f}")
print(f"Interpolation Probability      : {total_interp:.8f}")
print(f"Custom Discounted Bigram Prob. : {total_prob:.8f}")


================ FINAL COMPARISON =================

Unigram Probability            : 0.00000170
Bigram Probability             : 0.02777778
Trigram Probability            : 0.00000000
Interpolation Probability      : 0.01025085
Custom Discounted Bigram Prob. : 0.00002634
